# ramabana

> rama's arrow. a harness that does not miss.


ramabana is the **brain** of a coding agent: policy, tools, memory, and routing.
It has no editor and no frontend. The application speaks `Host`; models speak [rishi](https://github.com/vedicreader/rishi).

Built inside [leela](https://github.com/vedicreader/leela), now standalone. Ships a terminal app and an MCP server on the same `Host`.


## Features

| Capability | Primitive | What you get |
|---|---|---|
| Model runtime | [rishi](https://github.com/vedicreader/rishi) | LiteRT, MLX, llama.cpp, hosted FastLLM — one adapter |
| Code index | [kosha](https://github.com/vedicreader/kosha) | FTS + vectors + call graph over open folders and the env |
| Search fusion | [litesearch](https://github.com/vedicreader/litesearch) | RRF across kosha, ripgrep, and vault prose |
| Web research | [fossick](https://github.com/vedicreader/fossick) | search, fetch escalation, research digests, shopping carts |
| Durable memory | [vishalakshi](https://github.com/vedicreader/vishalakshi) | vault notes, federated search, watches |
| Fastai style | [fastcore](https://github.com/AnswerDotAI/fastcore) | `@delegates`, `L`, `parallel`, docments |

Routing sends cheap jobs (labels, summaries, compaction) to a small local model and the turn to whatever you chose. Tools the host cannot perform are never offered.


## Modules

One notebook per module — the page you read is the module you import.

| notebook | module | owns |
|---|---|---|
| `00_core` | `ramabana.core` | errors, env, routing |
| `01_runtime` | `ramabana.runtime` | rishi adapter, usage, compaction |
| `02_tools` | `ramabana.tools` | `Host`, `LocalHost`, tools, skills, sub-agents |
| `03_agent` | `ramabana.agent` | approvals, activity feed, `Agent` |
| `04_testing` | `ramabana.testing` | full host and backend doubles |
| `05_cli` | `ramabana.cli` | terminal app ([teleprint](https://github.com/answerdotai/teleprint)) |
| `06_mcp` | `ramabana.mcp` | same tools over MCP |
| `07_vault` | `ramabana.vault` | vault host on vishalakshi |
| `08_shop` | `ramabana.shop` | trolley on fossick |
| `09_coding_patterns` | `ramabana.coding_patterns` | Answer.AI / fastai coding standards |
| `10_spec` | `ramabana.spec` | OpenAPI / Discovery / GraphQL host |
| `11_pyrepl` | `ramabana.pyrepl` | `--python` kernel overlay ([dhrishti](https://github.com/vedicreader/dhrishti)) |


## Install

```sh
pip install ramabana                 # harness
pip install 'ramabana[cli]'          # + terminal app
pip install 'ramabana[pyrepl]'       # + Python prompt
pip install 'ramabana[mcp]'          # + MCP server
pip install 'ramabana[all]'
```

Models are not bundled. `rishi` fetches on first use; see [core](00_core.ipynb) for routing.
LiteRT defaults to GPU and falls back to CPU; pin with `RAMABANA_LITERT_BACKEND=cpu`.


## Use

`LocalHost` is a real-folder `Host`. Without a downloaded model the agent reports `ready=False` instead of raising.


In [ ]:
from ramabana import Agent
from ramabana.tools import LocalHost, tools_for

host = LocalHost(['..'], web=True)
agent = Agent(host, extensions=False)
len(agent.tools), agent.ready, agent.note


`LocalHost` starts `Kosha.sync` in a daemon thread over every open root. Search uses a literal fallback until the index is ready.


In [ ]:
#| eval: false
host.wait_index(120), host.search_note


In [ ]:
#| eval: false
[(h.path, h.line, h.symbol) for h in host.search('drop the thinking from a streamed reply')[:3]]


Capabilities the host lacks are omitted from the tool list — a partial host is a smaller agent, not a broken one:


In [ ]:
from ramabana.tools import NullHost
len(tools_for(NullHost())), len(tools_for(host))


`read_outside=True` allows reads anywhere on the machine; writes stay inside open folders. Credential paths are refused either way. Enumeration (`walk` / `grep` / `list_files`) never leaves the open roots.


In [ ]:
open_host = LocalHost(['..'], web=False, index=False, read_outside=True)
open_host.roots_note


One turn is `ask`. Against a scripted backend the page stays reproducible ([testing](04_testing.ipynb)):


In [ ]:
from ramabana.testing import fake_agent

scripted, backend = fake_agent(replies=['`threshold` is in `ramabana/runtime.py`.'])
scripted.ask('where is the compaction threshold?')


Afterwards: what it called, what it changed, what it cost.


In [ ]:
scripted.calls, scripted.changes(), repr(scripted.use)


## Surfaces

```sh
ramabana --root . --model gpt-mini          # terminal
ramabana --python --root .                  # Python prompt inside the same app
ramabana-mcp --root .                       # MCP server (read-only; --write for edits)
```

`--vault` keeps session reads in a vishalakshi vault. `--read_outside` matches the host flag above.
Details: [cli](05_cli.ipynb), [pyrepl](11_pyrepl.ipynb), [mcp](06_mcp.ipynb).


## Tour (live)

The hermetic cells above always run. The cells below need a model and the network; they are `eval: false` so CI skips them. Captured outputs are real.

Task: find the compactor output reserve in this repo, fetch Tim Tam Original 200g at Coles, then combine them. Routing keeps labels/summaries on a local 9B while the turn uses a hosted model.


In [ ]:
#| eval: false
from ramabana.agent import Agent
from ramabana.tools import LocalHost

host = LocalHost(['..'], web=True)
host.wait_index(600)
agent = Agent(host, model='gpt-mini', extensions=False)
for job in ('inline', 'completion', 'classify', 'summary', 'subagent'):
    agent.routing.set('ornith-9b', job)
agent.start() is not None, agent.note, len(agent.tools)


In [ ]:
#| eval: false
print(agent.routing.summary())
sorted(agent.routing.backends())


In [ ]:
#| eval: false
TASK = ("Two facts, then one line of arithmetic.\n"
        "1. In this repository, find the constant the compactor holds back for the model's reply, "
        "and say what it is called and what it is set to.\n"
        "2. Find what Arnott's Tim Tam Original 200g costs at Coles right now, in AUD.\n"
        "Finish with one line: the constant, the price, and how many packs those tokens would "
        "buy at $0.001 per token.")
answer = agent.ask(TASK)
print(answer)
for name, args in agent.calls: print(name, {k: str(v)[:64] for k, v in args.items()})
repr(agent.use), agent.changes(), agent.problems


`read_url` escalates thin 200-OK shells through fossick and keeps `schema.org` JSON-LD beside the prose — that is how the shelf price survives readability extraction.


In [ ]:
#| eval: false
import json
page = host.read_url("https://www.coles.com.au/product/arnott's-tim-tam-chocolate-biscuits-original-200g-329607")
ld = json.loads(page.text.partition('</structured-data>')[0].removeprefix('<structured-data>\n'))
ld[0]['name'], ld[0]['offers'][0]['price'], ld[0]['offers'][0]['priceCurrency']


In [ ]:
#| eval: false
agent.classify('the price came back from coles', ['success', 'failure'])
agent.summarise(answer)


## Develop

Notebooks in `nbs/` are the source. Do not edit generated modules.

```sh
uv sync --extra dev
uv run nbdev-export           # notebooks -> ramabana/*.py
uv run nbdev-test             # execute every notebook
uv run pytest                 # plain-python feature-block suite
uv run nbdev-clean            # before committing
```


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
